<a href="https://colab.research.google.com/github/SanjaraT/Langchain/blob/main/student_assistant_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain langchain-core langchain-community langchain-ollama

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import create_agent

# Database

In [6]:

students = {
    101: {"name": "Sanjara", "cgpa": 3.90},
    102: {"name": "Rahim", "cgpa": 3.75}
}

# Tools

In [7]:
@tool
def get_student(student_id: int):
    """Get information about a student using student ID."""

    return students.get(
        student_id,
        "Student not found."
    )


@tool
def add_student(
    student_id: int,
    name: str,
    cgpa: float
):
    """Add a new student."""

    students[student_id] = {
        "name": name,
        "cgpa": cgpa
    }

    return f"Student {name} added successfully."


@tool
def delete_student(student_id: int):
    """Delete a student using student ID."""

    if student_id in students:

        deleted = students.pop(student_id)

        return f"Deleted {deleted['name']}"

    return "Student not found."


@tool
def list_students():
    """List all students."""

    return students

In [8]:
tools = [
    get_student,
    add_student,
    delete_student,
    list_students
]

# Ollama

In [ ]:
# Install zstd dependency
!sudo apt-get install zstd -y

In [ ]:
# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [22]:
# Start Ollama server in the background
import subprocess
import time

# Set the OLLAMA_HOST environment variable
import os
os.environ['OLLAMA_HOST'] = '0.0.0.0'

# Start Ollama server in the background
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server some time to start
print("Waiting for Ollama server to start...")
time.sleep(10)
print("Ollama server started.")

Waiting for Ollama server to start...
Ollama server started.


In [23]:
!ollama pull llama3.1:8b

In [24]:
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)

# Agent

In [25]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful student assistant.

    You have access to tools for:
    - Getting student information
    - Adding students
    - Deleting students
    - Listing all students

    Always use the appropriate tool when the user asks about students.
    """
)

# Chat loop

In [26]:
print("Student Assistant Started")
print("Type 'exit' to stop\n")

while True:

    query = input("You: ")

    if query.lower() == "exit":
        break

    try:

        result = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": query
                    }
                ]
            }
        )

        print("\nAssistant:")

        # Print last AI message
        print(result["messages"][-1].content)

        print()

    except Exception as e:

        print(f"\nError: {e}\n")

Student Assistant Started
Type 'exit' to stop

You: Add a student named Karim with ID 103 and CGPA 3.85

Assistant:
```
{
    "message": "Student Karim added successfully.",
    "student_id": 103,
    "name": "Karim",
    "cgpa": 3.85
}
```

You: show all the student list

Assistant:
Here is the list of students:

1. Sanjara (101) - CGPA: 3.9
2. Rahim (102) - CGPA: 3.75
3. Karim (103) - CGPA: 3.85

You: exit
